In [5]:
import pandas as pd
seller_df = pd.read_csv('../csv-files/olist_sellers_dataset.csv')
geo_ref = pd.read_csv('../csv-files/geo_ref.csv')
seller_df.info()
geo_ref.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 6153 entries, 0 to 6152
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   UF              6153 non-null   str  
 1   ref_city        6153 non-null   str  
 2   ref_prefix_i    6153 non-null   int64
 3   ref_prefix_f    6153 non-null   int64
 4   ref_latitude    6067 non-null   str  
 5   ref_longiitude  6067 non-null   str  
 6   ref_state       6067 non-null   str  
dtypes: int64(2), str(5)
memory usage: 336.6 KB


In [4]:
seller_df.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [6]:
matches = seller_df['seller_city'].isin(geo_ref['ref_city'])
print(matches.sum())

3014


In [ ]:


clean_seller_df = seller_df.copy()

# 1. Clean reference data without deleting your secondary zip ranges
geo_ref_clean = geo_ref.copy()
geo_ref_clean['ref_city'] = geo_ref_clean['ref_city'].str.lower().str.strip()
geo_ref_clean['UF'] = geo_ref_clean['UF'].str.upper().str.strip()

# 2. Build a multi-range dictionary: { ('city', 'state'): [ (start1, end1), (start2, end2) ] }
geo_lookup = {}
for _, row in geo_ref_clean.iterrows():
    key = (row['ref_city'], row['UF'])
    range_tuple = (int(row['ref_prefix_i']), int(row['ref_prefix_f']))
    
    if key not in geo_lookup:
        geo_lookup[key] = []
    geo_lookup[key].append(range_tuple)

# 3. Standardize seller data
clean_seller_df['_temp_city'] = clean_seller_df['seller_city'].str.lower().str.strip()
clean_seller_df['_temp_state'] = clean_seller_df['seller_state'].str.upper().str.strip()

# 4. Check both city and state existence together
seller_city_state_tuples = list(zip(clean_seller_df['_temp_city'], clean_seller_df['_temp_state']))
combo_is_valid = [combo in geo_lookup for combo in seller_city_state_tuples]

# 5. Validate zip codes against ALL available ranges
zip_corrections = 0

def verify_and_align_zips_multi(row):
    global zip_corrections
    
    city_key = row['_temp_city']
    state_key = row['_temp_state']
    current_zip = int(row['seller_zip_code_prefix'])
    
    # Retrieve the list of all valid ranges for this specific city+state
    valid_ranges = geo_lookup[(city_key, state_key)]
    
    # Check if current zip code falls into ANY of the valid ranges
    is_zip_valid = any(start <= current_zip <= end for start, end in valid_ranges)
    
    # If it is not in ANY range, overwrite it with the absolute first valid zip of the first range
    if not is_zip_valid:
        absolute_base_zip = valid_ranges[0][0] # First range, starting zip
        row['seller_zip_code_prefix'] = absolute_base_zip
        zip_corrections += 1
        
    return row

# Execute the corrections safely on matching pairs
valid_combo_rows = clean_seller_df[combo_is_valid].apply(verify_and_align_zips_multi, axis=1)

# --- RECOMBINE AS NORMAL ---
manual_cleaning_bucket = clean_seller_df[~pd.Series(combo_is_valid, index=clean_seller_df.index)].copy()
final_processed_sellers = pd.concat([valid_combo_rows, manual_cleaning_bucket]).reset_index(drop=True)
final_processed_sellers.drop(columns=['_temp_city', '_temp_state'], inplace=True, errors='ignore')

print("==================================================")
print("   MULTI-RANGE STATE-AWARE PIPELINE REPORT       ")
print("==================================================")
print(f"Total Sellers Input:               {len(seller_df)}")
print(f"Rows with Valid City+State Pairs:  {len(valid_combo_rows)}")
print(f" -> Zip Codes Corrected for State: {zip_corrections}")
print("--------------------------------------------------")
print(f"🚨 Invalid Combo / Misspelled Bucket: {len(manual_cleaning_bucket)}")
print("==================================================")


   MULTI-RANGE STATE-AWARE PIPELINE REPORT       
Total Sellers Input:               3095
Rows with Valid City+State Pairs:  2980
 -> Zip Codes Corrected for State: 29
--------------------------------------------------
🚨 Invalid Combo / Misspelled Bucket: 115


In [40]:
manual_cleaning_bucket.info()

<class 'pandas.DataFrame'>
Index: 115 entries, 70 to 3036
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               115 non-null    str  
 1   seller_zip_code_prefix  115 non-null    int64
 2   seller_city             115 non-null    str  
 3   seller_state            115 non-null    str  
 4   temp_row_id             115 non-null    int64
 5   _temp_city              115 non-null    str  
 6   _temp_state             115 non-null    str  
dtypes: int64(2), str(5)
memory usage: 7.2 KB


In [38]:
manual_cleaning_bucket.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,temp_row_id,_temp_city,_temp_state
70,f410c8873029fcc3809b9df6d0b28914,95076,caxias do sul,SP,70,caxias do sul,SP
78,731ef20c231d9a7103a425e83fd91271,88501,lages - sc,SC,78,lages - sc,SC
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN,114,rio de janeiro,RN
182,5e49e3a44bdeb5aab2684258bbd4f525,88330,balenario camboriu,SC,182,balenario camboriu,SC
191,e88165a185134e13fdfc85d4fa654db8,8517,ferraz de vasconcelos,SP,191,ferraz de vasconcelos,SP


In [41]:
 #1. Take a fresh copy of your manual cleaning bucket
review_df = manual_cleaning_bucket.copy()

# 2. Merge with geo_ref using a range lookup
# We use a conditional cross-merge or a map to find the correct range
geo_mapped_failures = []

for idx, row in review_df.iterrows():
    z = int(row['seller_zip_code_prefix'])
    
    # Locate the true geographical row based on the math boundary posts
    match = geo_ref[(geo_ref['ref_prefix_i'] <= z) & (geo_ref['ref_prefix_f'] >= z)]
    
    if not match.empty:
        row['geo_ref_true_city'] = match.iloc[0]['ref_city']
        row['geo_ref_state'] = match.iloc[0]['UF']
    else:
        row['geo_ref_true_city'] = "ZIP OUT OF BOUNDS"
        row['geo_ref_state'] = "UNKNOWN"
        
    geo_mapped_failures.append(row)

# 3. Convert the results back into a clean, displayable DataFrame
comparison_df = pd.DataFrame(geo_mapped_failures)

# 4. Filter and select the exact columns to inspect side-by-side
ordered_review_columns = [
    'seller_id',
    'seller_zip_code_prefix',
    'seller_city',          # What the seller typed (The Typo)
    'geo_ref_true_city',    # What geo_ref says it should be (The Solution)
    'seller_state',         # What the seller typed
    'geo_ref_state'         # What geo_ref says it should be
]

# Display the entire table for your manual review
comparison_df[ordered_review_columns].tail(20)

,seller_id,seller_zip_code_prefix,seller_city,geo_ref_true_city,seller_state,geo_ref_state
2484,3078096983cf766a32a06257648502d1,13720,scao jose do rio pardo,sao jose do rio pardo,SP,sp
2510,cce6ab8d1682639fe45ab70234f1665f,81020,curitiba,curitiba,SP,pr
2536,0b18d63d0cd1d723567903fd34a07df2,5141,sp,sao paulo,SP,sp
2573,8f2ce03f928b567e3d56181ae20ae952,5141,pirituba,sao paulo,SP,sp
2589,3a52d63a8f9daf5a28f3626d7eb9bd28,71900,aguas claras df,brasilia,SP,df
2592,6601ee6383e7f452be71929f8de48bbb,14027,ribeirao pretp,ribeirao preto,SP,sp
2598,ec5c1d94df153d473b37f880977ae58e,21320,rio de janeiro,rio de janeiro,SP,rj
2620,6a3139c7bf09ece22a4713d956acbe5e,13790,sao sebastiao da grama/sp,sao sebastiao da grama,SP,sp
2646,7ad41305e96a6cab8294cd65891e9a86,28810,rio bonito,rio bonito,SP,rj
2649,24a6daf925d9d591870a66660416de31,14078,robeirao preto,ribeirao preto,SP,sp


In [42]:
import pandas as pd

# 1. Take a fresh copy of your manual cleaning bucket to repair
repaired_bucket_df = manual_cleaning_bucket.copy()

# Ensure integer tracking types across both dataframes for exact matching
repaired_bucket_df['seller_zip_code_prefix'] = repaired_bucket_df['seller_zip_code_prefix'].astype(int)
geo_ref_sorted = geo_ref.sort_values(by='ref_prefix_i').copy()
geo_ref_sorted['ref_prefix_i'] = geo_ref_sorted['ref_prefix_i'].astype(int)
geo_ref_sorted['ref_prefix_f'] = geo_ref_sorted['ref_prefix_f'].astype(int)

# Tracking metrics for your final report
successful_repaired_count = 0
unmappable_zip_count = 0

repaired_rows_list = []

# 2. Iterate and overwrite rows based on the zip code source of truth
for idx, row in repaired_bucket_df.iterrows():
    z = row['seller_zip_code_prefix']
    
    # Locate the math boundary posts matching this specific zip prefix
    match = geo_ref_sorted[(geo_ref_sorted['ref_prefix_i'] <= z) & (geo_ref_sorted['ref_prefix_f'] >= z)]
    
    if not match.empty:
        # OVERWRITE: Replace typos/wrong states with clean data from geo_ref
        row['seller_city'] = match.iloc[0]['ref_city']
        row['seller_state'] = match.iloc[0]['UF']
        successful_repaired_count += 1
    else:
        # Keep original row state intact if the zip code itself makes no sense
        unmappable_zip_count += 1
        
    repaired_rows_list.append(row)

# 3. Convert the repaired list back into a DataFrame
fully_cleaned_bucket_df = pd.DataFrame(repaired_rows_list)

# 4. RECOMBINE everything back into your absolute master seller dataset
# Merges the automatically validated cohort with this newly repaired cohort
final_master_sellers = pd.concat([valid_combo_rows, fully_cleaned_bucket_df]).reset_index(drop=True)

# Remove any temporary processing columns if they still exist
final_master_sellers.drop(columns=['_temp_city', '_temp_state'], inplace=True, errors='ignore')


# --- GENERATE YOUR AUTOMATION REPAIR REPORT ---
print("==================================================")
print("       AUTOMATED ZIP-HEALING PIPELINE REPORT      ")
print("==================================================")
print(f"Total Sellers Processed:           {len(seller_df)}")
print(f"Perfect Rows (Passed First Check): {len(valid_combo_rows)}")
print(f"Isolated Rows Handled via Zip Map: {len(repaired_bucket_df)}")
print("--------------------------------------------------")
print(f"✅ Typo Rows Successfully Overwritten: {successful_repaired_count}")
print(f"❌ Unmappable Rows (Zip Out of Bounds):  {unmappable_zip_count}")
print("--------------------------------------------------")
print(f"🎯 Total Cleaned Rows in Master Output: {len(final_master_sellers)}")
print("==================================================")


       AUTOMATED ZIP-HEALING PIPELINE REPORT      
Total Sellers Processed:           3095
Perfect Rows (Passed First Check): 2980
Isolated Rows Handled via Zip Map: 115
--------------------------------------------------
✅ Typo Rows Successfully Overwritten: 114
❌ Unmappable Rows (Zip Out of Bounds):  1
--------------------------------------------------
🎯 Total Cleaned Rows in Master Output: 3095


In [44]:
# The loop keeps unmappable rows completely untouched. 
# We can find the 1 escape by testing which row fails to match ANY valid range in geo_ref.

escaped_row = fully_cleaned_bucket_df[
    ~fully_cleaned_bucket_df['seller_zip_code_prefix'].apply(
        lambda z: any((geo_ref['ref_prefix_i'] <= int(z)) & (geo_ref['ref_prefix_f'] >= int(z)))
    )
]

# Display the true 1 culprit
escaped_row[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']]


,seller_id,seller_zip_code_prefix,seller_city,seller_state
1656,f00e21b1e91a79653163b7fd8f293ff1,37795,andradas,SP


In [45]:
# Directly overwrite the specific cells using the row index
final_master_sellers.at[1656, 'seller_zip_code_prefix'] = 37838
final_master_sellers.at[1656, 'seller_city'] = 'andradas'
final_master_sellers.at[1656, 'seller_state'] = 'MG'

# Quick validation check to confirm it changed
final_master_sellers.iloc[[1656]][['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']]


,seller_id,seller_zip_code_prefix,seller_city,seller_state
1656,989becdce12ebc39863c2bceab6f3ca1,37838,andradas,MG


In [53]:
final_master_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,temp_row_id
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,0
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,1
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,2
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,3
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,4


In [ ]:
final_master_sellers['seller_zip_code_prefix'] = final_master_sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

In [56]:
final_master_sellers = final_master_sellers.drop(columns='temp_row_id')


In [59]:
final_master_sellers['seller_state'] = final_master_sellers['seller_state'].str.lower()
final_master_sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,sp
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,sp
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,rj
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,sp
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,sp


In [60]:
final_master_sellers.to_csv('../df_cleaned/sellers_cln.csv')